# Clean F1-optimal thresholds for multiple models

This is the single threshold notebook for the black-box benchmark. Choose any registered models in MODELS. For each model, dataset, and category, it runs clean inference on the fixed evaluation IDs, selects the threshold that maximizes image F1, stores the scores, and packages all model artifacts together.

The selected operating point uses clean evaluation labels, following benchmark F1-max reporting. The evaluator freezes it for adversarial accuracy, flip rate, targeted success, FPR, and FNR.


In [ ]:
import hashlib
import shutil
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: SELECT MODELS, CLONE CODE, AND INSTALL DEPENDENCIES =====')

# Pick one or more registered adapters. Add future adapter names here after
# adding their repository/checkpoint configuration below.
MODELS = ('anomalyclip', 'aaclip', 'filo', 'afclip')
DATASETS = ('mvtec', 'visa')
BATCH_SIZE = 2
IMAGE_SIZE = 518

SUPPORTED_MODELS = {'anomalyclip', 'aaclip', 'filo', 'afclip'}
unknown = set(MODELS) - SUPPORTED_MODELS
if unknown:
    raise ValueError(f'Add setup/configuration for unsupported models: {sorted(unknown)}')
if not MODELS or len(set(MODELS)) != len(MODELS):
    raise ValueError('MODELS must contain one or more unique model names.')

WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
ANOMALYCLIP_ROOT = WORKING / 'AnomalyCLIP'
AACLIP_ROOT = WORKING / 'AA-CLIP'
FILO_ROOT = WORKING / 'FiLo'
AFCLIP_ROOT = WORKING / 'AF-CLIP'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
ANOMALYCLIP_REPO_URL = 'https://github.com/zqhang/AnomalyCLIP.git'
AACLIP_REPO_URL = 'https://github.com/Mwxinnn/AA-CLIP.git'
FILO_REPO_URL = 'https://github.com/CASIA-LMC-Lab/FiLo.git'
AFCLIP_REPO_URL = 'https://github.com/Faustinaqq/AF-CLIP.git'
ANOMALYCLIP_COMMIT = '3911738c0867544f545a076ad78f3f11d9ecbfdf'
AACLIP_COMMIT = '53db195f230442aa118c246876c94ba1c76139cc'
FILO_COMMIT = '36ff29ca09ba8ba3af24d7654582aea856031400'
AFCLIP_COMMIT = 'bb7edec4128a76f29cb573cd3002538bf250b2fe'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
if 'anomalyclip' in MODELS:
    clone_or_update(ANOMALYCLIP_REPO_URL, ANOMALYCLIP_ROOT, ANOMALYCLIP_COMMIT)
if 'aaclip' in MODELS:
    clone_or_update(AACLIP_REPO_URL, AACLIP_ROOT, AACLIP_COMMIT)
if 'filo' in MODELS:
    clone_or_update(FILO_REPO_URL, FILO_ROOT, FILO_COMMIT)
if 'afclip' in MODELS:
    clone_or_update(AFCLIP_REPO_URL, AFCLIP_ROOT, AFCLIP_COMMIT)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')
], check=True)
if 'filo' in MODELS:
    # Keep Kaggle's PyTorch; install only FiLo's additional runtime dependencies.
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'timm==0.9.16', 'transformers==4.38.1', 'addict==2.4.0',
        'yapf==0.40.2', 'prefetch_generator==1.0.3', 'huggingface-hub>=0.22',
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e',
        str(FILO_ROOT / 'models' / 'GroundingDINO'),
    ], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if {'aaclip', 'afclip'} & set(MODELS):
    import torch

    base_model_name = 'ViT-L-14-336px.pt'
    base_model_sha256 = '3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02'
    base_model_url = (
        'https://openaipublic.azureedge.net/clip/models/'
        + base_model_sha256 + '/' + base_model_name
    )
    base_model_cache = (
        AACLIP_ROOT / 'model' if 'aaclip' in MODELS
        else WORKING / 'afclip_clip_cache'
    )
    base_model_cache.mkdir(parents=True, exist_ok=True)
    base_model_path = base_model_cache / base_model_name
    attached_base = next(
        (path for path in Path('/kaggle/input').rglob(base_model_name) if path.is_file()),
        None,
    )
    if not base_model_path.is_file() or sha256(base_model_path) != base_model_sha256:
        if attached_base is not None:
            if sha256(attached_base) != base_model_sha256:
                raise RuntimeError(f'Attached base model has wrong SHA256: {attached_base}')
            shutil.copy2(attached_base, base_model_path)
        else:
            torch.hub.download_url_to_file(
                base_model_url,
                str(base_model_path),
                hash_prefix=base_model_sha256,
                progress=True,
            )
    if sha256(base_model_path) != base_model_sha256:
        raise RuntimeError(f'AA-CLIP base-model checksum mismatch: {base_model_path}')

print('Selected models:', MODELS)
print('Experiment code:', EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline')


In [ ]:
import torch

print('===== STEP 2: RESOLVE DATASETS AND MODEL CHECKPOINTS =====')

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
EVALUATION_INDEX = (
    EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'dataset_csv'
    / 'canonical_clip_per_dataset' / 'evaluation_test_indices.csv'
)
if not EVALUATION_INDEX.is_file():
    raise FileNotFoundError(f'Fixed evaluation CSV not found: {EVALUATION_INDEX}')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

MODEL_CONFIGURATIONS = {}

if 'anomalyclip' in MODELS:
    mvtec_checkpoint = (
        ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale' / 'epoch_15.pth'
    )
    visa_checkpoint = (
        ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale_visa' / 'epoch_15.pth'
    )
    for checkpoint in (mvtec_checkpoint, visa_checkpoint):
        if not checkpoint.is_file():
            available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
            raise FileNotFoundError(
                f'AnomalyCLIP checkpoint not found: {checkpoint}. Available: {available}'
            )
    MODEL_CONFIGURATIONS['anomalyclip'] = {
        'model_kwargs_by_target': {
            'mvtec': {
                'repository_root': str(ANOMALYCLIP_ROOT),
                'checkpoint_path': str(mvtec_checkpoint),
                'clip_download_root': str(WORKING / 'clip_cache'),
            },
            'visa': {
                'repository_root': str(ANOMALYCLIP_ROOT),
                'checkpoint_path': str(visa_checkpoint),
                'clip_download_root': str(WORKING / 'clip_cache'),
            },
        },
        'run_metadata': {
            'repository': ANOMALYCLIP_REPO_URL,
            'commit': ANOMALYCLIP_COMMIT,
            'zero_shot_mapping': {
                'mvtec': 'AnomalyCLIP MVTec target checkpoint',
                'visa': 'AnomalyCLIP VisA target checkpoint',
            },
        },
    }

if 'aaclip' in MODELS:
    checkpoint_dataset_url = (
        'https://www.kaggle.com/datasets/parsagh1383/aa-clip-checkpoints-main'
    )
    checkpoint_roots = [
        path for path in (
            Path('/kaggle/input/aa-clip-checkpoints-main'),
            Path('/kaggle/input/datasets/parsagh1383/aa-clip-checkpoints-main'),
        ) if path.is_dir()
    ]
    if not checkpoint_roots:
        raise FileNotFoundError(
            'Attach parsagh1383/aa-clip-checkpoints-main: ' + checkpoint_dataset_url
        )

    def resolve_training_checkpoint(training_name):
        matches = []
        for root in checkpoint_roots:
            for image_path in root.rglob('image_adapter.pth'):
                if training_name.lower() in {part.lower() for part in image_path.parts}:
                    matches.append(image_path.parent)
        matches = sorted(set(matches))
        if len(matches) != 1:
            raise RuntimeError(
                f'Expected one {training_name} checkpoint directory, found: {matches}'
            )
        directory = matches[0]
        image_path = directory / 'image_adapter.pth'
        text_path = directory / 'text_adapter.pth'
        return image_path, text_path if text_path.is_file() else None

    train_mvtec_image, train_mvtec_text = resolve_training_checkpoint('TrainOnMVTec')
    train_visa_image, train_visa_text = resolve_training_checkpoint('TrainOnVisA')
    MODEL_CONFIGURATIONS['aaclip'] = {
        'model_kwargs_by_target': {
            'mvtec': {
                'repository_root': str(AACLIP_ROOT),
                'image_checkpoint_path': str(train_visa_image),
                'text_checkpoint_path': (
                    str(train_visa_text) if train_visa_text else None
                ),
                'target_dataset': 'mvtec',
            },
            'visa': {
                'repository_root': str(AACLIP_ROOT),
                'image_checkpoint_path': str(train_mvtec_image),
                'text_checkpoint_path': (
                    str(train_mvtec_text) if train_mvtec_text else None
                ),
                'target_dataset': 'visa',
            },
        },
        'run_metadata': {
            'repository': AACLIP_REPO_URL,
            'commit': AACLIP_COMMIT,
            'checkpoint_dataset': checkpoint_dataset_url,
            'zero_shot_mapping': {
                'mvtec': 'TrainOnVisA',
                'visa': 'TrainOnMVTec',
            },
        },
    }

if 'filo' in MODELS:
    from huggingface_hub import hf_hub_download

    filo_checkpoint_repo = 'FantasticGNU/FiLo'
    filo_checkpoint_revision = '17bbc781de7c206fa0bb94c616ed895ca7bcc913'
    filo_checkpoint_cache = WORKING / 'filo_checkpoints'
    def released_filo_checkpoint(filename):
        return Path(hf_hub_download(
            repo_id=filo_checkpoint_repo, filename=filename,
            revision=filo_checkpoint_revision,
            local_dir=filo_checkpoint_cache,
        ))

    filo_train_mvtec = released_filo_checkpoint('filo_train_on_mvtec.pth')
    filo_train_visa = released_filo_checkpoint('filo_train_on_visa.pth')
    grounding_train_mvtec = released_filo_checkpoint('grounding_train_on_mvtec.pth')
    grounding_train_visa = released_filo_checkpoint('grounding_train_on_visa.pth')
    MODEL_CONFIGURATIONS['filo'] = {
        'model_kwargs_by_target': {
            'mvtec': {
                'repository_root': str(FILO_ROOT),
                'checkpoint_path': str(filo_train_visa),
                'grounding_checkpoint_path': str(grounding_train_visa),
                'target_dataset': 'mvtec',
            },
            'visa': {
                'repository_root': str(FILO_ROOT),
                'checkpoint_path': str(filo_train_mvtec),
                'grounding_checkpoint_path': str(grounding_train_mvtec),
                'target_dataset': 'visa',
            },
        },
        'run_metadata': {
            'repository': FILO_REPO_URL,
            'commit': FILO_COMMIT,
            'checkpoint_repository': filo_checkpoint_repo,
            'checkpoint_revision': filo_checkpoint_revision,
            'backbone': 'OpenAI ViT-L/14@336px',
            'zero_shot_mapping': {
                'mvtec': 'FiLo + Grounding DINO TrainOnVisA',
                'visa': 'FiLo + Grounding DINO TrainOnMVTec',
            },
        },
    }

if 'afclip' in MODELS:
    weight_root = AFCLIP_ROOT / 'weight'
    for checkpoint in (
        weight_root / 'mvtec_prompt.pt', weight_root / 'mvtec_adaptor.pt',
        weight_root / 'visa_prompt.pt', weight_root / 'visa_adaptor.pt',
    ):
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Released AF-CLIP checkpoint not found: {checkpoint}')
    MODEL_CONFIGURATIONS['afclip'] = {
        'model_kwargs_by_target': {
            'mvtec': {
                'repository_root': str(AFCLIP_ROOT),
                'prompt_checkpoint_path': str(weight_root / 'visa_prompt.pt'),
                'adaptor_checkpoint_path': str(weight_root / 'visa_adaptor.pt'),
                'clip_download_root': str(base_model_cache),
            },
            'visa': {
                'repository_root': str(AFCLIP_ROOT),
                'prompt_checkpoint_path': str(weight_root / 'mvtec_prompt.pt'),
                'adaptor_checkpoint_path': str(weight_root / 'mvtec_adaptor.pt'),
                'clip_download_root': str(base_model_cache),
            },
        },
        'run_metadata': {
            'repository': AFCLIP_REPO_URL,
            'commit': AFCLIP_COMMIT,
            'backbone': 'OpenAI ViT-L/14@336px',
            'zero_shot_mapping': {
                'mvtec': 'VisA prompt + adaptor weights',
                'visa': 'MVTec prompt + adaptor weights',
            },
        },
    }

print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)
print('Evaluation IDs:', EVALUATION_INDEX)
print('Configured models:', tuple(MODEL_CONFIGURATIONS))


In [ ]:
from blackbox_evaluation_pipeline import (
    ThresholdCalibrationConfig,
    calibrate_thresholds,
)

print('===== STEP 3: CALIBRATE CLEAN F1-OPTIMAL THRESHOLDS =====')
OUTPUT_ROOT = WORKING / 'f1_optimal_thresholds'
GENERATED_THRESHOLDS = {}

for model_name in MODELS:
    print(f'\n===== MODEL: {model_name} =====')
    model_setup = MODEL_CONFIGURATIONS[model_name]
    config = ThresholdCalibrationConfig(
        output_root=str(OUTPUT_ROOT / model_name),
        model_name=model_name,
        model_kwargs_by_target=model_setup['model_kwargs_by_target'],
        datasets=DATASETS,
        mvtec_root=str(MVTEC_ROOT),
        visa_root=str(VISA_ROOT),
        device='cuda',
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        evaluation_index_path=str(EVALUATION_INDEX),
        provenance='clean_evaluation_f1_optimal_following_crane',
        official_model_threshold=False,
        run_metadata={
            **model_setup['run_metadata'],
            'threshold_selection': 'maximize image F1 on fixed clean evaluation IDs',
            'threshold_scope': 'model x dataset x category',
            'frozen_for_adversarial_threshold_metrics': True,
        },
    )
    GENERATED_THRESHOLDS[model_name] = calibrate_thresholds(config)


In [ ]:
import json

print('===== STEP 4: PRINT GENERATED OPERATING POINTS =====')
for model_name, model_paths in GENERATED_THRESHOLDS.items():
    for dataset, threshold_path in model_paths.items():
        payload = json.loads(threshold_path.read_text(encoding='utf-8'))
        print(f'\n[{model_name} / {dataset}] {threshold_path}')
        print('category | threshold | F1-max | precision | recall | normal | anomaly')
        for category, record in payload['categories'].items():
            print(
                f"{category:12s} | {record['threshold']:.6f} | "
                f"{record['f1_max']:7.3f} | "
                f"{record['precision_at_threshold']:9.3f} | "
                f"{record['recall_at_threshold']:6.3f} | "
                f"{record['normal_count']:6d} | {record['anomaly_count']:7d}"
            )


In [ ]:
print('===== STEP 5: PACKAGE ALL SELECTED MODELS =====')
archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
print('Packaged thresholds:', archive)
print('Download this output and replace the matching repository thresholds after verification.')
